In [1]:
import pandas as pd
import numpy as np
import os
from scipy.signal import butter, filtfilt
import ast
from tqdm import tqdm

In [2]:
data_path = "C:\\Users\\Pranav Yeturu\\Desktop\\New folder\\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"  # <-- CHANGE THIS
csv_path = os.path.join(data_path, "ptbxl_database.csv")
scp_path = os.path.join(data_path, "scp_statements.csv")
records_path = os.path.join(data_path, "records500")  # contains .npy files

In [6]:
df = pd.read_csv(csv_path)
scp_df = pd.read_csv(scp_path)

# Convert scp_codes to dict
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

# Create mapping: SCP code → diagnostic class
print(scp_df.columns)
scp_diagnostic = scp_df[scp_df['diagnostic_class'].notnull()]
scp_map = dict(zip(scp_diagnostic['Unnamed: 0'], scp_diagnostic['diagnostic_class']))

def map_diagnostic(code_dict):
    return [scp_map[code] for code in code_dict if code in scp_map]

df['diagnostic_superclass'] = df['scp_codes'].apply(map_diagnostic)

Index(['Unnamed: 0', 'description', 'diagnostic', 'form', 'rhythm',
       'diagnostic_class', 'diagnostic_subclass', 'Statement Category',
       'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code',
       'DICOM Code'],
      dtype='object')


In [ ]:
common_classes = ['NORM', 'MI', 'STTC', 'CD', 'HYP', 'AF']
df['filtered_class'] = df['diagnostic_superclass'].apply(lambda x: [c for c in x if c in common_classes])
df = df[df['filtered_class'].map(len) > 0].reset_index(drop=True)
#Filtering for common classses or common disease groups

In [ ]:
def butter_highpass_filter(data, cutoff=0.5, fs=500, order=1):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, data, axis=0)

import wfdb

def load_ecg_wfdb(filename_hr, base_path, filter=True):
    try:
        record_path = os.path.join(base_path, filename_hr)
        record = wfdb.rdrecord(record_path)
        signal = record.p_signal

        if filter:
            signal = butter_highpass_filter(signal)
            signal = (signal - np.mean(signal, axis=0)) / (np.std(signal, axis=0) + 1e-8)

        return signal
    except Exception as e:
        print(f"⚠️ Could not load {filename_hr}: {e}")
        return None


In [ ]:
# 🧪 Apply Preprocessing to First N Samples
N = 1000  # Adjust based on memory
processed_signals = []
labels = []

for i in tqdm(range(N)):
    row = df.iloc[i]
    signal = load_ecg_wfdb(row['filename_hr'], base_path=data_path)
    if os.path.exists(filepath):
        signal = preprocess_signal(filepath)
        processed_signals.append(signal)
        labels.append(row['filtered_class'])

processed_signals = np.array(processed_signals)  # shape: [N, 5000, 12]


100%|██████████| 1000/1000 [00:00<00:00, 6849.39it/s]


In [10]:
# 💾 Save Preprocessed Dataset
np.save("ecg_signals_12lead.npy", processed_signals)
pd.DataFrame({'label': labels}).to_csv("ecg_labels.csv", index=False)

print("✅ Preprocessing complete! Shapes:")
print("Signals:", processed_signals.shape)
print("Labels:", len(labels))


✅ Preprocessing complete! Shapes:
Signals: (0,)
Labels: 0
